# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrishagk/My_flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
from getpass import getpass
import duckdb

HF_TOKEN = getpass("Enter your Hugging Face token: ")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

In [6]:

REL = "hf://datasets/FlyRank/internship-warehouse"

PERF_JAN = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2025-01/*.parquet'"
    f")"
)

print(con.sql(f"SELECT * FROM {PERF_JAN} LIMIT 5").df())

  report_date           client_hash_id           content_hash_id  \
0  2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
1  2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
2  2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   
3  2025-01-27  client_9958f0a7ae1df715  content_c899aef92518c714   
4  2025-01-27  client_9958f0a7ae1df715  content_c7c1d2e68d9d0964   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True                True               False   
1            True            True                True               False   
2            True            True                True               False   
3            True            True                True               False   
4            True            True                True               False   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               30           0               115  ...     

In [12]:
jan_sample = con.sql(f'SELECT * FROM {PERF_JAN}').df()

print(jan_sample.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [13]:
import numpy as np
import pandas as pd

# Start from the January performance data already loaded in ML-04.
df = jan_sample.copy()

# ---------------------------------------------------------
# 1. Raw numeric features available before the decision
# ---------------------------------------------------------
numeric_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_sum_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other",
    "scroll_events",
]

# ---------------------------------------------------------
# 2. Binary availability/context features
# ---------------------------------------------------------
binary_features = [
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
]

# Make sure numeric columns are numeric.
for col in numeric_features + binary_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------------------------------------------------------
# 3. Engineered features
# ---------------------------------------------------------

# CTR = clicks / impressions.
# Undefined when impressions are zero.
df["gsc_ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

# Engaged-session rate.
df["ga4_engagement_rate"] = np.where(
    df["ga4_sessions"] > 0,
    df["ga4_engaged_sessions"] / df["ga4_sessions"],
    np.nan
)

# Average engagement time per session.
df["avg_engagement_sec_per_session"] = np.where(
    df["ga4_sessions"] > 0,
    df["ga4_total_engagement_sec"] / df["ga4_sessions"],
    np.nan
)

# Organic traffic share.
total_non_null_sessions = (
    df["sessions_organic"]
    + df["sessions_direct"]
    + df["sessions_referral"]
    + df["sessions_social"]
    + df["sessions_paid"]
    + df["sessions_ai"]
)

df["organic_session_share"] = np.where(
    total_non_null_sessions > 0,
    df["sessions_organic"] / total_non_null_sessions,
    np.nan
)

# AI traffic share.
df["ai_session_share"] = np.where(
    total_non_null_sessions > 0,
    df["sessions_ai"] / total_non_null_sessions,
    np.nan
)

# ---------------------------------------------------------
# 4. Define the predictive feature set
# ---------------------------------------------------------
engineered_features = [
    "gsc_ctr",
    "ga4_engagement_rate",
    "avg_engagement_sec_per_session",
    "organic_session_share",
    "ai_session_share",
]

feature_columns = (
    numeric_features
    + binary_features
    + engineered_features
)

X = df[feature_columns].copy()

# ---------------------------------------------------------
# 5. Fill missing values
# ---------------------------------------------------------

# Binary availability fields:
# missing means the availability flag is not observed,
# so use 0 for this feature representation.
X[binary_features] = X[binary_features].fillna(0)

# Numeric features:
# use the median calculated from this available training snapshot.
numeric_X = [c for c in feature_columns if c not in binary_features]

for col in numeric_X:
    median_value = X[col].median()

    if pd.isna(median_value):
        median_value = 0

    X[col] = X[col].fillna(median_value)
# Final feature vector.
feature_vector = X

print("Feature matrix shape:", feature_vector.shape)
print("Number of features:", feature_vector.shape[1])
print("\nFeatures:")
print(feature_vector.columns.tolist())

print("\nRemaining missing values:")
print(feature_vector.isna().sum().sum())

Feature matrix shape: (1297, 32)
Number of features: 32

Features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'gsc_sum_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_ctr', 'ga4_engagement_rate', 'avg_engagement_sec_per_session', 'organic_session_share', 'ai_session_share']

Remaining missing values:
0


The feature vector uses metrics that can be observed before the decision point. I use the available GSC performance fields and binary data-availability fields, plus ratios that can be calculated from those same observations. Identifier fields are retained only for grouping and traceability, not as predictive features. Numeric missing values are filled with the training-partition median, while availability flags are filled with 0. Ratio features are left undefined when their denominator is zero and are then handled by the same numeric imputation step.

The current January partition has only five report dates, so this feature construction does not claim to provide a complete 90-day history. A production version should construct rolling historical features only after sufficient pre-decision coverage has been verified.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

GSC features: gsc_impressions, gsc_clicks, gsc_avg_position, and gsc_sum_position measure observed search performance. Missing numeric values are median-imputed. They are eligible only when the corresponding measurement was available before the decision point.

GA4 features: pageviews, sessions, users, engaged sessions, and engagement time measure observed site activity. They are median-imputed when missing. The January partition has GA4 unavailable, so these features should not be interpreted as observed GA4 performance for those rows.

Traffic-source features: organic, direct, referral, social, paid, and AI sessions describe observed traffic sources. Missing values are median-imputed; zero values are retained as observed values rather than automatically treated as missing.

AI-source features: ChatGPT, Perplexity, Gemini, Copilot, Claude, Meta, and other AI traffic fields describe observed AI-source traffic. They are available only when the underlying data exists before the decision point.

Scroll events: measures observed scrolling activity and is median-imputed if missing.

Availability flags: client_has_gsc, client_has_ga4, gsc_data_available, and ga4_data_available describe whether the corresponding data source is available. They are represented as binary features and missing flags are filled with 0.

Engineered features: CTR, engagement rate, average engagement time per session, organic-session share, and AI-session share are calculated only from fields in the same pre-decision observation. Division by zero produces an undefined value, which is subsequently median-imputed.

Identifiers and dates: client_hash_id, content_hash_id, report_date, and month are not predictive features. They are retained outside X for grouping, ordering, joining, and traceability.

Availability timing: a feature is eligible only if it could have been observed before the decision being predicted. The current January partition does not by itself prove a complete historical window, so rolling 30/90-day features should only be added after their coverage has been verified.

In [9]:
# Check that the feature vector contains only intended predictive fields.

print("Predictive features:")
for i, col in enumerate(feature_columns, start=1):
    print(f"{i:02d}. {col}")

print("\nFeature matrix:")
print("Rows:", len(feature_vector))
print("Columns:", len(feature_vector.columns))

print("\nMissing values after imputation:")
print(feature_vector.isna().sum().sum())

# Check that identifiers are NOT inside X.
identifier_columns = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
]

identifier_leak_check = [
    col for col in identifier_columns
    if col in feature_vector.columns
]

print("\nIdentifiers accidentally included as features:")
print(identifier_leak_check)

Predictive features:
01. gsc_impressions
02. gsc_clicks
03. gsc_avg_position
04. gsc_sum_position
05. ga4_pageviews
06. ga4_sessions
07. ga4_users
08. ga4_engaged_sessions
09. ga4_total_engagement_sec
10. sessions_organic
11. sessions_direct
12. sessions_referral
13. sessions_social
14. sessions_paid
15. sessions_ai
16. ai_chatgpt
17. ai_perplexity
18. ai_gemini
19. ai_copilot
20. ai_claude
21. ai_meta
22. ai_other
23. scroll_events
24. client_has_gsc
25. client_has_ga4
26. gsc_data_available
27. ga4_data_available
28. gsc_ctr
29. ga4_engagement_rate
30. avg_engagement_sec_per_session
31. organic_session_share
32. ai_session_share

Feature matrix:
Rows: 5
Columns: 32

Missing values after imputation:
20

Identifiers accidentally included as features:
[]


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the feature vector for three main leakage risks: identifier/label-derived fields, future-window fields, and fields whose availability could occur only after the decision. The predictive matrix does not contain client/content identifiers, report dates, the future outcome, or any explicitly named future-window metric.

The engineered features are calculated only from the selected observation fields and do not use the future outcome. The main remaining leakage risk is temporal rather than mathematical: a feature would be invalid if the underlying measurement was collected after the decision timestamp. Therefore, the final training pipeline must construct features using only rows whose observation time precedes the decision date.

In [10]:
# ---------------------------------------------------------
# Leakage test 1: forbidden identifiers / dates
# ---------------------------------------------------------

forbidden_columns = {
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
}

# Add common label/future-window naming patterns.
forbidden_patterns = [
    "label",
    "target",
    "outcome",
    "future",
    "post",
    "after",
    "next_",
    "30d",
    "90d",
]

leakage_candidates = []

for col in feature_vector.columns:
    col_lower = col.lower()

    if col in forbidden_columns:
        leakage_candidates.append((col, "identifier/date/context"))

    elif any(pattern in col_lower for pattern in forbidden_patterns):
        leakage_candidates.append((col, "possible label/future feature"))

print("Potential leakage candidates:")
print(leakage_candidates)

# ---------------------------------------------------------
# Leakage test 2: check that feature columns exist in the
# original January data and were not taken from a label table.
# ---------------------------------------------------------

original_columns = set(df.columns)

missing_from_source = [
    col for col in feature_columns
    if col not in original_columns
]

print("\nFeatures not present in source dataframe:")
print(missing_from_source)

# ---------------------------------------------------------
# Leakage test 3: confirm engineered features are derived
# only from allowed pre-decision source fields.
# ---------------------------------------------------------

engineered_dependencies = {
    "gsc_ctr": {"gsc_clicks", "gsc_impressions"},
    "ga4_engagement_rate": {
        "ga4_engaged_sessions",
        "ga4_sessions",
    },
    "avg_engagement_sec_per_session": {
        "ga4_total_engagement_sec",
        "ga4_sessions",
    },
    "organic_session_share": {
        "sessions_organic",
        "sessions_direct",
        "sessions_referral",
        "sessions_social",
        "sessions_paid",
        "sessions_ai",
    },
    "ai_session_share": {
        "sessions_ai",
        "sessions_organic",
        "sessions_direct",
        "sessions_referral",
        "sessions_social",
        "sessions_paid",
    },
}

allowed_source_fields = set(numeric_features + binary_features)

dependency_violations = {}

for engineered_feature, dependencies in engineered_dependencies.items():
    invalid_dependencies = dependencies - allowed_source_fields

    if invalid_dependencies:
        dependency_violations[engineered_feature] = invalid_dependencies

print("\nEngineered-feature dependency violations:")
print(dependency_violations)

# ---------------------------------------------------------
# Final assertion
# ---------------------------------------------------------

assert len(leakage_candidates) == 0
assert len(missing_from_source) == 0
assert len(dependency_violations) == 0

print("\nLeakage checks passed for the constructed feature vector.")

Potential leakage candidates:
[]

Features not present in source dataframe:
[]

Engineered-feature dependency violations:
{}

Leakage checks passed for the constructed feature vector.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [11]:
# Explicitly document fields that must not enter the model.

excluded_fields = {
    "client_hash_id": "identifier; may cause client memorization",
    "content_hash_id": "identifier; may cause content memorization",
    "report_date": "date/context rather than predictive measurement",
    "month": "partition/context field",
    "future outcome fields": "direct target leakage",
    "post-decision metrics": "not available at prediction time",
    "unclear-timing fields": "temporal leakage risk",
    "future rolling aggregates": "may contain post-decision information",
}

print("Excluded fields / field groups:")
for field, reason in excluded_fields.items():
    print(f"- {field}: {reason}")

# Final model matrix should contain no identifiers or dates.
assert "client_hash_id" not in feature_vector.columns
assert "content_hash_id" not in feature_vector.columns
assert "report_date" not in feature_vector.columns
assert "month" not in feature_vector.columns

print("\nExclusion checks passed.")

Excluded fields / field groups:
- client_hash_id: identifier; may cause client memorization
- content_hash_id: identifier; may cause content memorization
- report_date: date/context rather than predictive measurement
- month: partition/context field
- future outcome fields: direct target leakage
- post-decision metrics: not available at prediction time
- unclear-timing fields: temporal leakage risk
- future rolling aggregates: may contain post-decision information

Exclusion checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.